# AI/ML Methodology — Ballistic Target Identification
**Full training run on Kaggle GPU (Tesla T4 / P100)**

Steps:
1. Install dependencies
2. Define all modules inline (data, model, utils, train)
3. Run full training with 50 000 samples
4. Print accuracy + benchmark results

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q pytorch-lightning torchmetrics onnx onnxruntime tqdm

In [ ]:
# ── 2a. DATA MODULE ──────────────────────────────────────────────────────────
import os, math, pickle
import numpy as np
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm

T_MAX   = 60.0
DT      = 0.1
TIMESTEPS = int(T_MAX / DT)   # 600
TRACK_DIM = 4
BATCH     = 2_000
N_TRACKS  = 50_000
NOISE_STD = np.array([5, 3, 2e-3, 2e-3])
DATA_DIR  = '/kaggle/working/radar_data'
MODEL_DIR = '/kaggle/working/radar_models'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(MODEL_DIR + '/checkpoints', exist_ok=True)

MISSILE_SPECS = {
    0: dict(name='ATACMS',   v0_mu=1700, v0_sigma=80,  th_mu=45, th_sigma=3),
    1: dict(name='Iskander', v0_mu=2000, v0_sigma=100, th_mu=50, th_sigma=4),
    2: dict(name='GMLRS',    v0_mu=1300, v0_sigma=60,  th_mu=42, th_sigma=3),
}
g = 9.81

def ballistic_state(t, v0=1700., theta_deg=45.):
    theta = np.deg2rad(theta_deg)
    v0x, v0y = v0 * np.cos(theta), v0 * np.sin(theta)
    x  = v0x * t
    y  = v0y * t - 0.5 * g * t**2
    vx = np.full_like(t, v0x)
    vy = v0y - g * t
    return x, y, vx, vy

def range_doppler_snapshot(x, y, vx, vy):
    R  = np.hypot(x, y) + 1e-6
    vr = (x * vx + y * vy) / R
    az = np.arctan2(x, y)
    el = np.arctan2(y, x)
    return np.stack([R, vr, az, el], axis=-1)

def simulate_track(v0, theta):
    t = np.arange(0, T_MAX, DT)
    x, y, vx, vy = ballistic_state(t, v0, theta)
    snap  = range_doppler_snapshot(x, y, vx, vy)
    noise = np.random.normal(0, NOISE_STD, snap.shape)
    for i in range(1, len(noise)):
        noise[i] = 0.95 * noise[i-1] + 0.05 * noise[i]
    snap += noise
    return snap.astype(np.float32)

def generate_all(n_tracks=N_TRACKS):
    print(f'Generating {n_tracks} tracks in batches of {BATCH}...')
    all_data, all_labels, all_targets = [], [], []
    for b_idx in tqdm(range(0, n_tracks, BATCH)):
        actual = min(BATCH, n_tracks - b_idx)
        data, labels, targets = [], [], []
        for _ in range(actual):
            m_id = np.random.choice(list(MISSILE_SPECS))
            s    = MISSILE_SPECS[m_id]
            v0   = np.random.normal(s['v0_mu'], s['v0_sigma'])
            th   = np.random.normal(s['th_mu'], s['th_sigma'])
            trk  = simulate_track(v0, th)
            idx  = np.arange(TIMESTEPS) + 5
            idx[idx >= TIMESTEPS] = TIMESTEPS - 1
            targets.append(trk[idx, 0])
            data.append(trk)
            labels.append(m_id)
        bn = b_idx // BATCH
        np.save(f'{DATA_DIR}/data_{bn:03d}.npy',   np.stack(data))
        np.save(f'{DATA_DIR}/label_{bn:03d}.npy',  np.array(labels))
        np.save(f'{DATA_DIR}/target_{bn:03d}.npy', np.stack(targets))
        all_data.extend(data); all_labels.extend(labels); all_targets.extend(targets)
    all_data    = np.stack(all_data)
    all_targets = np.stack(all_targets)
    norm_stats = {
        'data_mean':    np.mean(all_data,    axis=(0, 1)),
        'data_std':     np.std(all_data,     axis=(0, 1)),
        'target_mean':  float(np.mean(all_targets)),
        'target_std':   float(np.std(all_targets)),
    }
    with open(f'{DATA_DIR}/norm_stats.pkl', 'wb') as f:
        pickle.dump(norm_stats, f)
    print(f'✅ Dataset generated: {len(all_data)} samples')
    return norm_stats

def ensure_data_exists(n_tracks=N_TRACKS):
    files    = [f for f in os.listdir(DATA_DIR) if f.startswith('data_') and f.endswith('.npy')]
    expected = math.ceil(n_tracks / BATCH)
    if len(files) != expected or not os.path.exists(f'{DATA_DIR}/norm_stats.pkl'):
        print(f'Regenerating {n_tracks} tracks...')
        for item in os.listdir(DATA_DIR):
            os.remove(os.path.join(DATA_DIR, item))
        generate_all(n_tracks)
    else:
        print(f'Found {len(files)} data files, data is ready.')

class RadarDS(Dataset):
    def __init__(self, split='train', normalize=True, n_tracks=N_TRACKS):
        all_files = sorted([f for f in os.listdir(DATA_DIR) if f.startswith('data_') and f.endswith('.npy')])
        n         = len(all_files)
        split_idx = max(1, int(0.8 * n))
        self.data_files = all_files[:split_idx] if split == 'train' else all_files[split_idx:]
        self.file_sizes       = [np.load(f'{DATA_DIR}/{f}', mmap_mode='r').shape[0] for f in self.data_files]
        self.cumulative_sizes = np.cumsum(self.file_sizes)
        self.total_samples    = int(self.cumulative_sizes[-1]) if len(self.cumulative_sizes) > 0 else 0
        print(f'{split} dataset: {self.total_samples} samples from {len(self.data_files)} files.')
        self.normalize = normalize
        if normalize:
            with open(f'{DATA_DIR}/norm_stats.pkl', 'rb') as f:
                self.norm_stats = pickle.load(f)

    def __len__(self): return self.total_samples

    def __getitem__(self, idx):
        file_idx   = np.searchsorted(self.cumulative_sizes, idx, side='right')
        local_idx  = idx - (self.cumulative_sizes[file_idx - 1] if file_idx > 0 else 0)
        file_name  = self.data_files[file_idx]
        file_num   = int(file_name.split('_')[1].split('.')[0])
        data       = np.load(f'{DATA_DIR}/data_{file_num:03d}.npy',   mmap_mode='r')[local_idx]
        label      = np.load(f'{DATA_DIR}/label_{file_num:03d}.npy',  mmap_mode='r')[local_idx]
        target     = np.load(f'{DATA_DIR}/target_{file_num:03d}.npy', mmap_mode='r')[local_idx]
        data   = torch.from_numpy(np.array(data,   dtype=np.float32))
        target = torch.from_numpy(np.array(target, dtype=np.float32))
        label  = torch.tensor(int(label), dtype=torch.long)
        if self.normalize:
            ns     = self.norm_stats
            mean   = torch.tensor(ns['data_mean'],   dtype=torch.float32)
            std    = torch.tensor(ns['data_std'],    dtype=torch.float32)
            data   = (data - mean) / (std + 1e-8)
            target = (target - ns['target_mean']) / (ns['target_std'] + 1e-8)
        return data, label, target

print('✅ Data module ready')

In [ ]:
# ── 2b. MODEL MODULE ─────────────────────────────────────────────────────────
import math
import pytorch_lightning as pl
import torch.nn as nn
import torch.nn.functional as F
from torchmetrics import Accuracy

class MCDropoutNet(pl.LightningModule):
    def __init__(self, dropout_rate=0.25, hidden_dim=512, mc_samples=25, lr=5e-4):
        super().__init__()
        self.save_hyperparameters()
        self.mc_samples   = mc_samples
        self.dropout_rate = dropout_rate
        self.lr           = lr

        # Expanded CNN backbone: 4 → 64 → 128 → 256 channels
        self.conv = nn.Sequential(
            nn.Conv1d(4, 64, 7, padding=3),    nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Conv1d(64, 128, 5, padding=2),  nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Conv1d(128, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
        )
        # BiLSTM: 256-dim input (matches CNN output), hidden_dim//2 per direction
        self.rnn = nn.LSTM(256, hidden_dim // 2, num_layers=2, batch_first=True,
                           bidirectional=True, dropout=dropout_rate)
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=8, batch_first=True, dropout=dropout_rate)
        self.norm = nn.LayerNorm(hidden_dim)  # stabilise pooled representation

        self.cls_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(), nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim // 2, 3))
        self.reg_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(), nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim // 2, 600))

        self.train_acc  = Accuracy(task='multiclass', num_classes=3)
        self.val_acc    = Accuracy(task='multiclass', num_classes=3)
        self.cls_weight = nn.Parameter(torch.tensor(1.0))
        self.reg_weight = nn.Parameter(torch.tensor(0.1))

    def forward(self, x):
        x = x.transpose(1, 2)                               # (B, 4, T)
        x = self.conv(x)                                     # (B, 256, T)
        x = x.transpose(1, 2)                               # (B, T, 256)
        rnn_out, _ = self.rnn(x)                             # (B, T, hidden_dim)
        attn_out, _ = self.attention(rnn_out, rnn_out, rnn_out)
        pooled = self.norm(torch.mean(attn_out, dim=1))      # (B, hidden_dim)
        return self.cls_head(pooled), self.reg_head(pooled)

    def _compute_loss(self, batch):
        x, y_true, z_true = batch
        cls_out, reg_out  = self(x)
        # label_smoothing=0.1 improves generalisation on near-separable classes
        cls_loss = F.cross_entropy(cls_out, y_true, label_smoothing=0.1)
        reg_loss = F.mse_loss(reg_out, z_true)
        cls_w = F.softplus(self.cls_weight)
        reg_w = F.softplus(self.reg_weight)
        return cls_w * cls_loss + reg_w * reg_loss, cls_out, cls_loss, reg_loss

    def training_step(self, batch, batch_idx):
        loss, cls_out, cls_loss, reg_loss = self._compute_loss(batch)
        _, y_true, _ = batch
        self.log('train_loss',     loss,     on_step=False, on_epoch=True, prog_bar=True)
        self.log('train_cls_loss', cls_loss, on_step=False, on_epoch=True)
        self.log('train_reg_loss', reg_loss, on_step=False, on_epoch=True)
        self.log('train_acc',      self.train_acc(cls_out, y_true), on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, cls_out, cls_loss, reg_loss = self._compute_loss(batch)
        _, y_true, _ = batch
        self.log('val_loss',     loss,     on_step=False, on_epoch=True, prog_bar=True)
        self.log('val_cls_loss', cls_loss, on_step=False, on_epoch=True)
        self.log('val_reg_loss', reg_loss, on_step=False, on_epoch=True)
        self.log('val_acc',      self.val_acc(cls_out, y_true), on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)

        def warmup_cosine(epoch):
            """3-epoch linear warmup then cosine decay to 5% of peak LR."""
            warmup = 3
            total  = self.trainer.max_epochs
            if epoch < warmup:
                return (epoch + 1) / warmup
            progress = (epoch - warmup) / max(1, total - warmup)
            return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))

        sch = torch.optim.lr_scheduler.LambdaLR(opt, warmup_cosine)
        return {'optimizer': opt, 'lr_scheduler': {'scheduler': sch, 'interval': 'epoch'}}

    def mc_predict(self, x, n_samples=None):
        """Bayesian inference: activate Dropout only — BatchNorm stays in eval mode."""
        n_samples = n_samples or self.mc_samples
        self.eval()
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.train()
        cls_preds, reg_preds = [], []
        with torch.no_grad():
            for _ in range(n_samples):
                c, r = self.forward(x)
                cls_preds.append(F.softmax(c, dim=-1))
                reg_preds.append(r)
        self.eval()  # restore full eval mode
        cls_stack   = torch.stack(cls_preds)
        reg_stack   = torch.stack(reg_preds)
        cls_mean    = cls_stack.mean(0)
        cls_entropy = -torch.sum(cls_mean * torch.log(cls_mean + 1e-9), dim=-1)
        return {'cls_mean': cls_mean, 'cls_std': cls_stack.std(0), 'cls_entropy': cls_entropy,
                'reg_mean': reg_stack.mean(0), 'reg_std': reg_stack.std(0)}

print('✅ Model module ready')

In [ ]:
# ── 2c. UTILS MODULE ─────────────────────────────────────────────────────────
import time
import onnx
import onnxruntime as ort

providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']

def export_to_onnx(model, save_path):
    model.eval()
    device      = next(model.parameters()).device
    dummy_input = torch.randn(1, 600, 4, device=device)
    try:
        torch.onnx.export(
            model, dummy_input, save_path,
            export_params=True, opset_version=14, do_constant_folding=True,
            input_names=['input'], output_names=['classification', 'regression'],
            dynamic_axes={'input': {0: 'batch_size'}, 'classification': {0: 'batch_size'}, 'regression': {0: 'batch_size'}}
        )
        onnx.checker.check_model(onnx.load(save_path))
        print(f'✅ ONNX exported & verified: {save_path}')
    except Exception as e:
        print(f'❌ ONNX export failed: {e}')
    return save_path

def benchmark_onnx_model(onnx_path, test_loader, n_runs=100):
    session  = ort.InferenceSession(onnx_path, providers=providers)
    x_sample = next(iter(test_loader))[0].numpy()
    for _ in range(10): session.run(None, {'input': x_sample})
    times = []
    for _ in tqdm(range(n_runs), desc='ONNX Benchmark'):
        t0 = time.perf_counter()
        session.run(None, {'input': x_sample})
        times.append(time.perf_counter() - t0)
    avg = np.mean(times) * 1000
    print(f'✅ ONNX Inference: {avg:.2f} ± {np.std(times)*1000:.2f} ms')
    return avg, np.std(times) * 1000

def comprehensive_benchmark(model, test_loader):
    print('🔍 Running comprehensive benchmark...')
    model.eval()
    device = next(model.parameters()).device
    cls_preds, reg_preds, cls_true, reg_true = [], [], [], []
    pytorch_times, mc_times = [], []
    with open(f'{DATA_DIR}/norm_stats.pkl', 'rb') as f:
        ns = pickle.load(f)
    with torch.no_grad():
        for x, y, z in tqdm(test_loader, desc='Benchmark'):
            x, y, z = x.to(device), y.to(device), z.to(device)
            t0 = time.perf_counter(); yp, zp = model(x); pytorch_times.append(time.perf_counter() - t0)
            t0 = time.perf_counter(); mc = model.mc_predict(x); mc_times.append(time.perf_counter() - t0)
            cls_preds.append(yp.argmax(-1).cpu()); reg_preds.append(zp.cpu())
            cls_true.append(y.cpu()); reg_true.append(z.cpu())
    cp = torch.cat(cls_preds); ct = torch.cat(cls_true)
    rp = torch.cat(reg_preds) * ns['target_std'] + ns['target_mean']
    rt = torch.cat(reg_true)  * ns['target_std'] + ns['target_mean']
    acc  = (cp == ct).float().mean().item()
    rmse = torch.sqrt(torch.mean((rp - rt) ** 2)).item()
    mae  = torch.mean(torch.abs(rp - rt)).item()
    results = {
        'classification_accuracy': acc, 'regression_rmse': rmse, 'regression_mae': mae,
        'pytorch_inference_ms_per_batch': np.mean(pytorch_times) * 1000,
        'mc_dropout_inference_ms_per_batch': np.mean(mc_times) * 1000,
    }
    print(f'Classification Accuracy: {acc:.4f}'); print(f'Regression RMSE: {rmse:.2f} m')
    return results

def test_noise_robustness(model, n_samples=500):
    print('🔬 Testing noise robustness...')
    device = next(model.parameters()).device
    model.eval()
    with open(f'{DATA_DIR}/norm_stats.pkl', 'rb') as f: ns = pickle.load(f)
    data_mean = torch.tensor(ns['data_mean'], dtype=torch.float32)
    data_std  = torch.tensor(ns['data_std'],  dtype=torch.float32)
    results = {}
    for mult in [0.5, 1.0, 1.5, 2.0, 3.0]:
        xs, ys = [], []
        for _ in range(n_samples):
            m_id = np.random.choice(list(MISSILE_SPECS))
            s    = MISSILE_SPECS[m_id]
            trk  = simulate_track(np.random.normal(s['v0_mu'], s['v0_sigma']),
                                   np.random.normal(s['th_mu'], s['th_sigma']))
            trk  = trk + np.random.normal(0, NOISE_STD * mult, trk.shape).astype(np.float32)
            t    = torch.from_numpy(trk).float()
            xs.append((t - data_mean) / (data_std + 1e-8)); ys.append(m_id)
        X = torch.stack(xs).to(device)
        Y = torch.tensor(ys, dtype=torch.long, device=device)
        with torch.no_grad():
            acc = (model(X)[0].argmax(-1) == Y).float().mean().item()
        results[mult] = {'acc': acc}
        print(f'  {mult}× noise → accuracy = {acc:.4f}')
    return results

print('✅ Utils module ready')

In [ ]:
# ── 3. GENERATE DATA ─────────────────────────────────────────────────────────
pl.seed_everything(42)
ensure_data_exists(N_TRACKS)
print('Target classes:', [s['name'] for s in MISSILE_SPECS.values()])

In [ ]:
# ── 4. DATALOADERS ───────────────────────────────────────────────────────────
from torch.utils.data import DataLoader

# On Kaggle: 2 CPU workers are safe; pin_memory only when CUDA is available
device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pin_memory  = device.type == 'cuda'
num_workers = 2

train_loader = DataLoader(RadarDS('train', n_tracks=N_TRACKS), batch_size=128,
                          shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
val_loader   = DataLoader(RadarDS('val',   n_tracks=N_TRACKS), batch_size=256,
                          num_workers=num_workers, pin_memory=pin_memory)

print(f'Device: {device}')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
# ── 5. TRAIN ─────────────────────────────────────────────────────────────────
model = MCDropoutNet(dropout_rate=0.25, hidden_dim=512, mc_samples=25, lr=5e-4)

early_stop = pl.callbacks.EarlyStopping(monitor='val_loss', patience=8, min_delta=0.001, mode='min')
checkpoint = pl.callbacks.ModelCheckpoint(
    monitor='val_acc', dirpath=MODEL_DIR + '/checkpoints',
    filename='radar-best', save_top_k=1, mode='max'
)
lr_monitor = pl.callbacks.LearningRateMonitor(logging_interval='epoch')

precision = '16-mixed' if device.type == 'cuda' else '32-true'

trainer = pl.Trainer(
    max_epochs=50,
    accelerator='auto',
    devices='auto',
    precision=precision,
    callbacks=[early_stop, checkpoint, lr_monitor],
    log_every_n_steps=10,
    gradient_clip_val=1.0,
)

print('🔥 Training model...')
trainer.fit(model, train_loader, val_loader)

In [ ]:
# ── 6. EVALUATE + BENCHMARK ──────────────────────────────────────────────────
best_path = checkpoint.best_model_path
if best_path:
    print(f'Loading best model from: {best_path}')
    model = MCDropoutNet.load_from_checkpoint(best_path).to(device)
else:
    print('No checkpoint saved — using last model state.')
    model = model.to(device)

onnx_path       = MODEL_DIR + '/radar_model.onnx'
export_to_onnx(model, onnx_path)
benchmark_res   = comprehensive_benchmark(model, val_loader)
onnx_avg, _     = benchmark_onnx_model(onnx_path, val_loader)
noise_res       = test_noise_robustness(model)

print('\n🎯 FINAL SUMMARY:')
print('=' * 60)
print(f"✅ Best Validation Accuracy : {benchmark_res['classification_accuracy']:.4f}")
print(f"✅ Regression RMSE          : {benchmark_res['regression_rmse']:.2f} m")
print(f"✅ PyTorch inference        : {benchmark_res['pytorch_inference_ms_per_batch']:.2f} ms/batch")
print(f"✅ MC-Dropout inference     : {benchmark_res['mc_dropout_inference_ms_per_batch']:.2f} ms/batch")
print(f"✅ ONNX inference           : {onnx_avg:.2f} ms/batch")
print('✅ Noise Robustness:')
for lvl, r in noise_res.items():
    print(f'   {lvl}× noise → {r["acc"]:.4f}')
print('=' * 60)